In [1]:
###############################################################################
# Open up the connection to pangeo
###############################################################################
from matplotlib import pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import dask
from dask.diagnostics import progress
from tqdm.autonotebook import tqdm
import intake
import fsspec
import seaborn as sns
import sys
from collections import defaultdict
import nc_time_axis
import fsspec
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = 12, 6
# open up the full pangeo set of data
pangeo = intake.open_esm_datastore("https://storage.googleapis.com/cmip6/pangeo-cmip6.json")


x = pangeo.df.loc[(pangeo.df['source_id'] == 'MIROC6') & (pangeo.df['variable_id'] == 'tas') & (pangeo.df['table_id'] == 'Amon')].copy()

# SSP245 is where we see that data from our Tgav calculations stops at 2039 for all but 3 realizations.
# Is it our Tgav code or the data? Check the data for one of the realizations that stops in 2039 in 
# our files.
x1 = x[x['experiment_id'] == 'ssp245'].copy()
x1 = x1[x1['member_id'] == 'r10i1p1f1'].copy()


print(x1.zstore.values[0])

/Users/snyd535/miniconda3/envs/pangeo/lib/python3.7/site-packages/ipykernel_launcher.py:10: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  # Remove the CWD from sys.path while we load stuff.


gs://cmip6/CMIP6/ScenarioMIP/MIROC/MIROC6/ssp245/r10i1p1f1/Amon/tas/gn/v20200623/


In [2]:
# open up that file and take a look what years of data are actually included in the netcdf
ds = xr.open_zarr(fsspec.get_mapper(x1.zstore.values[0]), consolidated=True)
ds

<xarray.Dataset>
Dimensions:    (bnds: 2, lat: 128, lon: 256, time: 300)
Coordinates:
    height     float64 ...
  * lat        (lat) float64 -88.93 -87.54 -86.14 -84.74 ... 86.14 87.54 88.93
    lat_bnds   (lat, bnds) float64 dask.array<chunksize=(128, 2), meta=np.ndarray>
  * lon        (lon) float64 0.0 1.406 2.812 4.219 ... 354.4 355.8 357.2 358.6
    lon_bnds   (lon, bnds) float64 dask.array<chunksize=(256, 2), meta=np.ndarray>
  * time       (time) datetime64[ns] 2015-01-16T12:00:00 ... 2039-12-16T12:00:00
    time_bnds  (time, bnds) datetime64[ns] dask.array<chunksize=(300, 2), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    tas        (time, lat, lon) float32 dask.array<chunksize=(150, 128, 256), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   60265.0
    branch_time_in_parent:  60265.0
    cmor_version:           3.5.0
    creation_date:          2020-06-12T16:59:37Z
    data_specs_version:     01.00.31
    experiment:             update of RCP4.5 based on SSP2
    experiment_id:          ssp245
    external_variables:     areacella
    forcing_index:          1
    frequency:              mon
    further_info_url:       https://furtherinfo.es-doc.org/CMIP6.MIROC.MIROC6...
    grid:                   native atmosphere T85 Gaussian grid
    grid_label:             gn
    history:                2020-06-12T16:59:37Z ; CMOR rewrote data to be co...
    initialization_index:   1
    institution:            JAMSTEC (Japan Agency for Marine-Earth Science an...
    institution_id:         MIROC
    license:                CMIP6 model data produced by MIROC is licensed un...
    mip_era:                CMIP6
    nominal_resolution:     250 km
    parent_activity_id:     CMIP
    parent_experiment_id:   historical
    parent_mip_era:         CMIP6
    parent_source_id:       MIROC6
    parent_time_units:      days since 1850-1-1
    parent_variant_label:   r10i1p1f1
    physics_index:          1
    product:                model-output
    realization_index:      10
    realm:                  atmos
    source:                 MIROC6 (2017): \naerosol: SPRINTARS6.0\natmos: CC...
    source_id:              MIROC6
    source_type:            AOGCM AER
    status:                 2020-07-02;created; by gcs.cmip6.ldeo@gmail.com
    sub_experiment:         none
    sub_experiment_id:      none
    table_id:               Amon
    table_info:             Creation Date:(22 July 2019) MD5:b4cefb4b6dbb146f...
    title:                  MIROC6 output prepared for CMIP6
    tracking_id:            hdl:21.14100/711021d0-8230-4d4b-a305-589faf6d0696
    variable_id:            tas
    variant_label:          r10i1p1f1
    netcdf_tracking_ids:    hdl:21.14100/711021d0-8230-4d4b-a305-589faf6d0696
    version_id:             v20200623

In [3]:
ds.time.values.max()

numpy.datetime64('2039-12-16T12:00:00.000000000')